## why the device needs monitoring: train on january, serve july

our device keeps a **30-day rolling buffer**. so at any moment the model was trained on roughly one month of data. what happens a few months later?

this notebook runs that experiment on real data and answers three things the rest of the project depends on:

1. how badly does a stale model fail, and **when** does it start?
2. **why** does it fail - the mechanism matters, because it decides the fix
3. what would monitoring have seen, and **how early**?

the punchline up front: the failure is not gradual, it is not fixable with hyperparameters, and the earliest signal arrives **before any ground truth does**.

In [ ]:
import sys, warnings
sys.path.insert(0, ".."); warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb

from streamflow.config import CONFIG, resolve
from streamflow.features import feature_columns, TARGET_COLUMN as T

sienna, olive, steel, gray = "#a0522d", "#808000", "#4682b4", "#b0b0b0"
plt.rcParams["axes.facecolor"] = "#faf7f2"; plt.rcParams["figure.facecolor"] = "#faf7f2"

feat = pd.read_parquet(resolve(CONFIG["data"]["features_path"]))
COLS = feature_columns(feat)

def nse(obs, pred):
    return 1 - ((obs-pred)**2).sum() / ((obs-obs.mean())**2).sum()

def fit(df):
    m = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                         verbosity=0, random_state=42)
    return m.fit(df[COLS], df[T])

jan = feat[(feat.time >= "2026-01-01") & (feat.time < "2026-02-01")]
jul = feat[feat.time >= "2026-07-01"].copy()
print(f"train (january): {len(jan):,} hourly rows")
print(f"serve (july):    {len(jul):,} hourly rows")

january yields fewer than 744 rows because the gauge iced over and stopped reporting hourly - the same data-quality issue we monitor for. the device really would have had a thin month.

### 1. how bad is it
three models on the same july test set. **nse below 0 means worse than predicting the average** - the model is not just weak, it is harmful.

In [ ]:
m_jan = fit(jan)
m_all = fit(feat[feat.time < "2026-07-01"])

y = jul[T].values
preds = {
    "trained on january": m_jan.predict(jul[COLS]),
    "trained on all history": m_all.predict(jul[COLS]),
    "persistence baseline": np.log1p(jul["flow_max_24h"].values),
}
pd.DataFrame([{"model": k, "NSE": round(nse(y, p), 3),
               "log-RMSE": round(float(np.sqrt(((y-p)**2).mean())), 3)}
              for k, p in preds.items()]).set_index("model")

### 2. when does it break
not on day one. the january model is **fine for two weeks**, then fails and never recovers. this is the trap: a weekly accuracy review would have called it healthy.

In [ ]:
jul["pred"] = preds["trained on january"]
jul["bias"] = jul["pred"] - jul[T]
jul["pred_cfs"] = np.expm1(jul["pred"]); jul["true_cfs"] = np.expm1(jul[T])

daily = jul.set_index("time").resample("D").agg(
    predicted=("pred_cfs", "mean"), actual=("true_cfs", "mean"), bias=("bias", "mean"))
daily.round(1)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True,
                       gridspec_kw={"height_ratios": [2, 1]})
ax[0].plot(jul.time, jul.true_cfs, color=steel, lw=1.2, label="actual")
ax[0].plot(jul.time, jul.pred_cfs, color=sienna, lw=1.2, ls="--", label="january model")
ax[0].set_yscale("log"); ax[0].set_ylabel("next-24h peak (cfs)")
ax[0].set_title("january model serving july", weight="bold"); ax[0].legend(fontsize=8)

ax[1].axhline(0, color=gray, lw=1)
ax[1].fill_between(jul.time, 0, jul.bias, color=olive, alpha=.6)
ax[1].set_ylabel("bias (log units)"); ax[1].set_xlabel("")
for a in ax: a.axvline(pd.Timestamp("2026-07-14", tz="UTC"), color=sienna, ls=":", lw=1.5)
ax[1].annotate("collapse", xy=(pd.Timestamp("2026-07-14", tz="UTC"), 1.5),
               fontsize=9, color=sienna)
plt.tight_layout(); plt.show()

### 3. why it breaks
two measurements explain everything.

**(a) the output collapses to a single value.** the model stops responding to its inputs entirely.

In [ ]:
late = jul[(jul.time >= "2026-07-15") & (jul.time < "2026-07-26")]
print(f"july 15-25, {len(late)} hourly predictions")
print(f"  predicted: min {late.pred_cfs.min():.0f}  max {late.pred_cfs.max():.0f}  std {late.pred_cfs.std():.2f} cfs")
print(f"  actual:    min {late.true_cfs.min():.0f}  max {late.true_cfs.max():.0f}  std {late.true_cfs.std():.2f} cfs")
print("\n-> the model emits a constant while the river recedes by a third.")

**(b) july is outside the feature space january ever saw.** trees cannot extrapolate: an input beyond the training range falls into whatever leaf the splits happen to reach, and every such input reaches the *same* leaf. that is the constant above.

note this is a property of tree models, not a tuning mistake - see the hyperparameter note at the end.

In [ ]:
rows = []
for c in COLS:
    lo, hi = jan[c].min(), jan[c].max()
    outside = ((jul[c] < lo) | (jul[c] > hi)).mean() * 100
    if outside > 0:
        rows.append({"feature": c, "january range": f"[{lo:,.1f}, {hi:,.1f}]",
                     "july outside %": round(outside, 1)})
outside_df = pd.DataFrame(rows).sort_values("july outside %", ascending=False)
print(f"{len(outside_df)} of {len(COLS)} features see values january never contained\n")
outside_df.head(8).to_string(index=False)

### 4. what monitoring sees, and when
this is the payoff. the four signals fire at **different times**, and the most useful one needs no ground truth at all.

In [ ]:
# input drift: share of features outside the training range, per day - no labels needed
lo, hi = jan[COLS].min(), jan[COLS].max()
oob = (((jul[COLS] < lo) | (jul[COLS] > hi)).mean(axis=1) * 100)
jul["oob_pct"] = oob.values

daily2 = jul.set_index("time").resample("D").agg(
    input_oob_pct=("oob_pct", "mean"),
    pred_std=("pred_cfs", "std"),
    abs_bias=("bias", lambda s: s.abs().mean()))

fig, ax = plt.subplots(3, 1, figsize=(11, 6.5), sharex=True)
ax[0].fill_between(daily2.index, 0, daily2.input_oob_pct, color=sienna, alpha=.7)
ax[0].set_ylabel("% features\nout of range"); ax[0].set_title("1. input drift - available on day one, no ground truth", weight="bold", fontsize=10)
ax[1].fill_between(daily2.index, 0, daily2.pred_std, color=olive, alpha=.7)
ax[1].set_ylabel("prediction\nstd (cfs)"); ax[1].set_title("2. prediction drift - variance collapses, still no ground truth", weight="bold", fontsize=10)
ax[2].fill_between(daily2.index, 0, daily2.abs_bias, color=steel, alpha=.7)
ax[2].set_ylabel("|bias|\n(log units)"); ax[2].set_title("3. performance drift - needs the actual value, arrives last", weight="bold", fontsize=10)
plt.tight_layout(); plt.show()

print(f"input drift on july 1:        {daily2.input_oob_pct.iloc[0]:.0f}% of features already out of range")
print(f"performance drift by july 14: |bias| {daily2.abs_bias.loc['2026-07-14']:.2f} log units")

**the lesson for our design:** input drift is visible immediately; error-based monitoring only speaks once the next 24 hours have elapsed. if we watched accuracy alone we would have shipped two weeks of bad forecasts. that is why the monitor checks inputs, predictions, *and* errors.

### 5. the part that is easy to get wrong: this is drift, the flood is not
both look like "big error". they demand opposite responses, so the monitor has to tell them apart.

In [ ]:
good = fit(feat[feat.time < "2026-06-01"])          # a healthy, current model
flood = feat[(feat.time >= "2026-07-01") & (feat.time < "2026-07-08")].copy()
flood["err"] = np.abs(good.predict(flood[COLS]) - flood[T])

print("healthy model through the flood - does the error persist?")
print(flood.set_index("time").resample("D")["err"].mean().round(2).to_string())
print("\n-> error spikes on the 2nd, then returns to baseline within days: RECOVERS = anomaly")
print("-> the january model's error rises and stays: PERSISTS = drift")

| | flood (2 jul) | staleness (14 jul on) |
|---|---|---|
| error | spikes | rises and stays |
| recovery | days | never |
| inputs | extreme but in-range | outside training range |
| cause | rare event | the world moved |
| **response** | **flag, widen interval, do NOT retrain** | **retrain** |

retraining on the flood would teach the model that 3,000 cfs is normal, using the rarest data we have. our monitor separates them with a **fast window** (48h, catches spikes, never triggers retraining) and a **slow window** (30d, needs consecutive breaches, the only thing that does).

### 6. so what fixes it
not hyperparameters. the january model never saw a summer - no configuration invents data. below, the same failure at three very different settings.

In [ ]:
settings = {"shallow, heavy reg": dict(max_depth=2, min_child_weight=20, reg_lambda=10),
            "default":            dict(max_depth=6, min_child_weight=1,  reg_lambda=1),
            "deep, light reg":    dict(max_depth=10, min_child_weight=1, reg_lambda=0)}
out = []
for name, kw in settings.items():
    m = xgb.XGBRegressor(n_estimators=300, learning_rate=0.05, verbosity=0,
                         random_state=42, **kw).fit(jan[COLS], jan[T])
    out.append({"hyperparameters": name, "NSE on july": round(nse(y, m.predict(jul[COLS])), 3)})

buffer_like = feat[(feat.time >= "2026-06-01") & (feat.time < "2026-07-01")]   # a fresh 30-day buffer
out.append({"hyperparameters": "default, RETRAINED on june",
            "NSE on july": round(nse(y, fit(buffer_like).predict(jul[COLS])), 3)})
pd.DataFrame(out).set_index("hyperparameters")

every january configuration fails - shallow or deep, heavy or light regularisation, all land between nse -1.5 and -1.8. **refitting the same configuration on recent data is what actually helps** (-1.83 -> +0.30).

the honest limit: +0.30 is a repair, not a triumph - the full-history model reaches 0.69. one month of data is thin no matter how fresh it is. that is exactly why retraining uses **buffer + golden archive** rather than the buffer alone.

so "retraining" in our pipeline means refit on current data with a fixed configuration, not a hyperparameter search. tuning still matters, but it happens **once, offline, by a human**, and its goal is a configuration that stays sane when refit on data nobody has seen yet. the automatic retrain reuses that fixed configuration so every retrain is fast, deterministic and reproducible.

### takeaways
- a 30-day buffer goes stale in about two weeks, then fails **catastrophically** (nse -1.8, worse than the mean)
- the mechanism is extrapolation past the training range, so the model emits a constant - a tree property, not a tuning bug
- **input drift is visible on day one**; error-based monitoring only speaks after the fact - we need both
- anomaly vs drift is decided by whether the error **recovers**; only drift may trigger retraining
- the fix is fresher data (buffer + golden archive), which is exactly what the retrain path does